In [2]:
import os    # путь к файлу
import zstandard as zstd    # декомпрессор
import tarfile    # распаковка архива
import glob    # поиск файлов по шаблону

from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import xarray as xr
from tqdm.notebook import tqdm

from joblib import Parallel, delayed, cpu_count    # распараллеливание

import math

import warnings

from skimage.measure import label, regionprops
from scipy.optimize import linear_sum_assignment

In [3]:
path = '.'

# Работа с архивом исходных масок (mask_cycl_2d)

In [4]:
# Читает файлы из папки для создания выборки
def get_samples_from_folder(folder_path, max_samples=100):
    samples = []
    for root, dirs, files in os.walk(folder_path):
        for file in files[:max_samples]:
            with open(os.path.join(root, file), 'rb') as f:
                samples.append(f.read())
    return samples


# Обучение словаря
def train_zstd_dictionary(folder_with_samples, dict_path, dict_size=112640):
    samples = get_samples_from_folder(folder_with_samples)
    
    dict_data = zstd.train_dictionary(dict_size, samples)
    
    with open(dict_path, 'wb') as f:
        f.write(dict_data.as_bytes())
    print(f"Словарь сохранен в: {dict_path}")


# Сжатие с использованием словаря
def compress_with_dict(source_dir, output_archive, dict_path):
    # Загружаем словарь
    with open(dict_path, 'rb') as d:
        dict_data = zstd.ZstdCompressionDict(d.read())
    
    # Создаем компрессор со словарем
    cctx = zstd.ZstdCompressor(dict_data=dict_data, level=3)
    
    with open(output_archive, 'wb') as f:
        with cctx.stream_writer(f) as compressor:
            with tarfile.open(fileobj=compressor, mode='w|') as tar:
                tar.add(source_dir, arcname=os.path.basename(source_dir))
    print(f"Архив создан: {output_archive}")


# Распаковка с использованием словаря
def decompress_with_dict(archive_path, extract_path, dict_path):
    os.makedirs(extract_path, exist_ok=True)
    
    # Загружаем словарь
    with open(dict_path, 'rb') as d:
        dict_data = zstd.ZstdCompressionDict(d.read())
    
    # Создаем декомпрессор со словарем
    dctx = zstd.ZstdDecompressor(dict_data=dict_data)
    
    with open(archive_path, 'rb') as f:
        with dctx.stream_reader(f) as decompressor:
            with tarfile.open(fileobj=decompressor, mode='r|') as tar:
                tar.extractall(path=extract_path)
    print(f"Архив распакован в: {extract_path}")

In [5]:
# Сжатие архива (если необходимо)
# train_zstd_dictionary(
#     os.path.join(path, 'mask_cycl_2d'),
#     os.path.join(path, 'dict_cycl_2d.dict')
# )
# compress_with_dict(
#     os.path.join(path, 'mask_cycl_2d'),
#     os.path.join(path, 'mask_cycl_2d.tar.zst'),
#     os.path.join(path, 'dict_cycl_2d.dict')
# )

In [6]:
# Распаковка архива
# decompress_with_dict(
#     os.path.join(path, 'mask_cycl_2d.tar.zst'),
#     os.path.join(path, 'mask_cycl_2d'),
#     os.path.join(path, 'dict_cycl_2d.dict')
# )

# Работа с архивом исходных данных (era_raw)

In [7]:
# Сжатие архива (если необходимо)
# train_zstd_dictionary(
#     os.path.join(path, 'era5_raw'),
#     os.path.join(path, 'dict_era5_raw.dict')
# )
# compress_with_dict(
#     os.path.join(path, 'era5_raw'),
#     os.path.join(path, 'era5_raw.tar.zst'),
#     os.path.join(path, 'dict_era5_raw.dict')
# )

In [8]:
# Распаковка архива
# decompress_with_dict(
#     os.path.join(path, 'era5_raw.tar.zst'),
#     os.path.join(path, 'era5_raw'),
#     os.path.join(path, 'dict_era5_raw.dict')
# )

Разбиение оригинальных данных ERA5 (1 файл = 1 год * все слои) на мелкие части (1 файл = 1 временная метка * 1 слой)

In [9]:
def process_era5_to_levels(input_file, base_output_dir):
    """
    Разбивает годовой файл ERA5 на отдельные файлы по времени и уровням давления.
    """
    target_levels = [300, 500, 600, 700, 850, 925, 1000]    # уровни
    
    # Открываем файл с "ленивой" загрузкой
    ds = xr.open_dataset(input_file)    # , chunks={'time': 1}
    print(f"Начинаю обработку файла {input_file}")

    os.makedirs(base_output_dir, exist_ok=True)

    for level in target_levels:
        # Создаем папку для конкретного уровня
        level_dir = os.path.join(base_output_dir, f"lvl_{level}")
        # if not os.path.exists(level_dir):
        #     os.makedirs(level_dir)
        os.makedirs(level_dir, exist_ok=True)
            
        # Выбираем срез по уровню
        ds_level = ds.sel({'level': level})

        for time_step in tqdm(ds_level.time.values, desc=f"Lvl {level}"):
            # Выбираем срез по времени
            ds_slice = ds_level.sel(time=time_step)
            
            # Формируем метку времени для имени файла
            ts = xr.DataArray(time_step).dt
            
            filename = f"era5_geopotential_{level}_{ts.year.values}_{ts.month.values:02d}_{ts.day.values:02d}_{ts.hour.values:02d}.nc"
            output_path = os.path.join(level_dir, filename)
            
            # Сохраняем (используем netcdf4, он быстрее для маленьких файлов)
            ds_slice.to_netcdf(output_path)

    print(f"Заканчиваю обработку файла {input_file}")

In [10]:
# for year in range(1979, 2023 + 1):
#     process_era5_to_levels(os.path.join(path, 'era5_raw', f'era5_geopotential.{year}.interp.nc'))

# Работа с архивом исходных данных (era_processed)

In [11]:
# Сжатие архива (если необходимо)
# train_zstd_dictionary(
#     os.path.join(path, 'era5_processed'),
#     os.path.join(path, 'dict_era5_processed.dict')
# )
# compress_with_dict(
#     os.path.join(path, 'era5_processed'),
#     os.path.join(path, 'era5_processed.tar.zst'),
#     os.path.join(path, 'dict_era5_processed.dict')
# )

In [12]:
# Распаковка архива
# decompress_with_dict(
#     os.path.join(path, 'era5_processed.tar.zst'),
#     os.path.join(path, 'era5_processed'),
#     os.path.join(path, 'dict_era5_processed.dict')
# )

# Вспомогательные функции

## Названия файлов

In [13]:
def create_filename_2d(dt, level):
    """
    Создает название файла по datetime и уровню

    Parameters:
    dt (datetime): объект datetime
    level (int): уровень (например, 1000, 850, 500)

    Returns:
    str: название файла в формате mask_cycl_level_year_month_day_hour.npy
    """
    # Форматируем компоненты даты с ведущими нулями где необходимо
    year = dt.year
    month = dt.month
    day = dt.day
    hour = dt.hour

    # Создаем название файла
    filename = f"mask_cycl_{level}_{year}_{month:02d}_{day:02d}_{hour:02d}.npy"

    return filename

In [14]:
def create_era5_filename(dt, level):
    """
    Создает название файла данных ERA5 по datetime и уровню.

    Parameters:
    dt (datetime): объект datetime
    level (int): уровень давления (например, 300, 500)

    Returns:
    str: название файла в формате era5_geopotential_level_year_month_day_hour.nc
    """
    year = dt.year
    month = dt.month
    day = dt.day
    hour = dt.hour

    filename = f"era5_geopotential_{level}_{year}_{month:02d}_{day:02d}_{hour:02d}.nc"
    
    return filename

In [15]:
def create_result_filename(level, start_dt, end_dt):
    """
    Создает название файла для сохранения результатов трекинга на основе временного интервала.
    
    Parameters:
    level (int): уровень давления (например, 300)
    start_dt (datetime): дата начала обработки
    end_dt (datetime): дата конца обработки

    Returns:
    str: название файла в формате tracks_level_YYYYMMDDHH_YYYYMMDDHH.txt
    """
    filename = ""

    if (start_dt == datetime(1979, 1, 1, 0, 0)) and (end_dt == datetime(2023, 12, 31, 18, 0)):
        filename = f"tracks_{level}.txt"
    else:
        # Форматируем даты в строку YYYY_MM_DD_HH для компактности
        start_str = start_dt.strftime("%Y_%m_%d_%H")
        end_str = end_dt.strftime("%Y_%m_%d_%H")
        
        filename = f"tracks_{level}_from_{start_str}_to_{end_str}.txt"
    
    return filename

In [16]:
def create_processed_track_filename(level, track_id):
    """
    Создает название файла для одного конкретного трека.
    """
    return f"track_{level}_{int(track_id):06d}.txt"

## Работа с масками

In [17]:
def create_bit_masks(arr):
    """
    Функция создаёт битовые маски отдельных циклонов на основе общей 2d маски. Самая быстрая версия - создаем все маски сразу.
    """
    unique_values = np.unique(arr)
    return {val: (arr == val).astype(np.uint8) for val in unique_values if val != 0}

## Получение информации о состоянии циклонов

In [18]:
def get_cyclones_at_step(mask_path, era5_path, level):
    """
    Извлекает характеристики всех циклонов для одного момента времени.
    
    Parameters:
        mask_path (str): Путь к .npy файлу маски
        era5_path (str): Путь к .nc файлу ERA5
        level (int): Текущий уровень давления
        
    Returns:
        list: Список словарей с признаками каждого найденного циклона
    """
    # 1. Загрузка маски
    if not os.path.exists(mask_path):
        return []
    
    mask = np.load(mask_path)
    # Если маска 3D (уровни, lat, lon), берем слой, соответствующий вашему порядку уровней
    # В вашем ноутбуке уровни: [1000, 925, 850, 700, 600, 500, 300]
    # Если файл содержит все уровни, нужно выбрать индекс. 
    # Если вы сохраняли по одному уровню, используем mask[0] или squeeze.
    if mask.ndim == 3:
        # Предполагаем, что для 2D трекинга мы загружаем файл, где нужный слой первый
        mask = mask[0]
    mask = mask.squeeze()

    # 2. Загрузка данных ERA5
    if not os.path.exists(era5_path):
        return []
    
    ds = xr.open_dataset(era5_path)
    
    # Определяем имя переменной (z для геопотенциала, msl для давления)
    var_name = 'z' #if 'z' in ds.data_vars else 'msl'
    data = ds[var_name].values.squeeze()
    
    lats = ds.latitude.values
    lons = ds.longitude.values
    ds.close()

    # 3. Сегментация объектов
    labeled_mask = label(mask)
    # intensity_image позволяет regionprops автоматически брать значения из ERA5
    props = regionprops(labeled_mask, intensity_image=data)
    
    step_features = []
    
    for prop in props:
        # --- Геометрический центр ---
        # Координаты в пикселях (y, x)
        yi, xi = prop.centroid
        # Интерполяция для получения точных координат между узлами сетки
        lat_cent = float(np.interp(yi, np.arange(len(lats)), lats))
        lon_cent = float(np.interp(xi, np.arange(len(lons)), lons))
        
        # --- Поиск минимума внутри маски ---
        coords = prop.coords # Список всех индексов [y, x], входящих в циклон
        intensities = data[coords[:, 0], coords[:, 1]]
        
        min_idx = np.argmin(intensities)
        y_min_idx, x_min_idx = coords[min_idx]
        
        val_min = float(intensities[min_idx])
        lat_min = float(lats[y_min_idx])
        lon_min = float(lons[x_min_idx])
        
        # --- Физические параметры ---
        # Глубина: разница между макс. и мин. значением внутри маски
        depth = float(prop.max_intensity - prop.min_intensity)
        
        # Радиус: эквивалентный радиус круга в градусах
        # Эквивалентный диаметр в пикселях * шаг сетки (0.25 для ERA5)
        grid_step = abs(lats[1] - lats[0])
        radius_deg = (prop.equivalent_diameter / 2) * grid_step
        
        step_features.append({
            'lat_min': lat_min,
            'lon_min': lon_min,
            'lat_cent': lat_cent,
            'lon_cent': lon_cent,
            'val_min': val_min,
            'depth': depth,
            'radius': radius_deg
        })
        
    return step_features

## Расчёт стоимости сопоставления циклонов

In [19]:
def calculate_cost(cyclone_prev, cyclone_curr, max_dist=1200):
    """
    Вычисляет штраф (стоимость) связи между двумя состояниями циклона.
    """
    # Расстояние между центрами (с учетом периодичности долготы)
    lat1, lon1 = cyclone_prev['lat_cent'], cyclone_prev['lon_cent']
    lat2, lon2 = cyclone_curr['lat_cent'], cyclone_curr['lon_cent']
    
    dlon = abs(lon1 - lon2)
    if dlon > 180: dlon = 360 - dlon
    
    # 1 градус ~ 111 км (упрощенно для скорости)
    dx = dlon * 111 * np.cos(np.radians((lat1 + lat2) / 2))
    dy = (lat1 - lat2) * 111
    dist_km = np.sqrt(dx**2 + dy**2)
    
    if dist_km > max_dist:
        return 1e9  # Физически невозможная связь
    
    # Разница интенсивности (val_min) и размера (radius)
    # Коэффициенты 0.01 и 10 подбираются под масштаб данных
    intensity_diff = abs(cyclone_prev['val_min'] - cyclone_curr['val_min']) * 0.01
    radius_diff = abs(cyclone_prev['radius'] - cyclone_curr['radius']) * 10
    
    return dist_km + intensity_diff + radius_diff

## Формат вывода

In [20]:
def format_output_line(dt, cyclone_id, info):
    """Подготовка строки для текстового файла."""
    return (f"{dt.year} {dt.month:02d} {dt.day:02d} {dt.hour:02d} "
            f"{int(cyclone_id)} "
            f"{info['lat_min']:.2f} {info['lon_min']:.2f} "
            f"{info['lat_cent']:.2f} {info['lon_cent']:.2f} "
            f"{info['val_min']:.2f} {info['depth']:.2f} {info['radius']:.2f}\n")

# Собственно трекинг

In [21]:
def run_time_tracking(start_dt, end_dt, level=300):
    """
    Запуск трекинга циклонов по времени для одного уровня.
    """
    # Подготовка файлов
    results_dir = os.path.join(path, "time_tracks_raw")
    os.makedirs(results_dir, exist_ok=True)

    results_file = create_result_filename(level, start_dt, end_dt)
    results_path = os.path.join(results_dir, results_file)
    
    # Инициализация
    current_time = start_dt
    next_id = 1
    active_tracks = [] # Список циклонов с предыдущего шага с их ID
    
    with open(results_path, 'w') as f:
        # Цикл по времени (от T до End с шагом 6ч)
        while current_time <= end_dt:
            # 1. Загружаем данные текущего шага
            mask_fn = os.path.join(path, 'mask_cycl_2d', create_filename_2d(current_time, level))
            era5_fn = os.path.join(path, 'era5_processed', f'lvl_{level}', create_era5_filename(current_time, level))
            
            if not (os.path.exists(mask_fn) and os.path.exists(era5_fn)):
                print(f"Skip {current_time}: Files not found.")
                current_time += timedelta(hours=6)
                continue
            
            # 2. Получаем параметры всех циклонов в этот момент
            current_cyclones = get_cyclones_at_step(mask_fn, era5_fn, level)
            
            # 3. Сопоставление с предыдущим шагом
            new_active_tracks = []
            
            if not active_tracks:
                # Первый шаг или после "разрыва": все циклоны получают новые ID
                for cyc in current_cyclones:
                    cyc['id'] = next_id
                    next_id += 1
                    new_active_tracks.append(cyc)
            else:
                # Строим матрицу стоимостей (active_tracks vs current_cyclones)
                costs = np.zeros((len(active_tracks), len(current_cyclones)))
                for i, prev_cyc in enumerate(active_tracks):
                    for j, curr_cyc in enumerate(current_cyclones):
                        costs[i, j] = calculate_cost(prev_cyc, curr_cyc)
                
                # Решаем задачу о назначениях
                row_ind, col_ind = linear_sum_assignment(costs)
                
                matched_current_indices = set()
                
                # # Назначаем ID для совпавших пар
                # for r, c in zip(row_ind, col_ind):
                #     if costs[r, c] < 1e9:
                #         current_cyclones[c]['id'] = active_tracks[r]['id']
                #         matched_current_indices.add(c)
                
                # # Для всех несовпавших — создаем новые ID
                # for j, cyc in enumerate(current_cyclones):
                #     if j not in matched_current_indices:
                #         cyc['id'] = next_id
                #         next_id += 1
                #     new_active_tracks.append(cyc)

                for r, c in zip(row_ind, col_ind):
                    if costs[r, c] < 1e9:
                        current_cyclones[c]['id'] = active_tracks[r]['id']
                        matched_current_indices.add(c)
                    # else: do nothing — will get a new ID below
                
                for j, cyc in enumerate(current_cyclones):
                    if j not in matched_current_indices:
                        cyc['id'] = next_id
                        next_id += 1
                    new_active_tracks.append(cyc)

            # 4. Запись результатов в файл
            for cyc in new_active_tracks:
                f.write(format_output_line(current_time, cyc['id'], cyc))
            
            # Подготовка к следующему шагу
            active_tracks = new_active_tracks
            current_time += timedelta(hours=6)
            
    print(f"Tracking complete. Results saved to: {results_file}")

##  С распараллеливанием

In [22]:
levels = [1000, 925, 850, 700, 600, 500, 300]

start_time = datetime(1979, 1, 1, 0, 0)    #### datetime(1979, 1, 1, 0, 0)
end_time = datetime(1989, 12, 31, 18, 0)    #### datetime(2023, 12, 31, 18, 0)

In [23]:
n_jobs = max(1, cpu_count() - 1)  # оставить 1 ядро системе
print(n_jobs)

7


In [24]:
start = datetime.now()
print(start)

Parallel(n_jobs=min(len(levels), n_jobs))(    # занимаем максимально возможное количество ядер (в моём случае 7 из 8)
        delayed(run_time_tracking)(start_time, end_time, lvl) for lvl in levels
    )

finish = datetime.now()
print(finish)

programm_length = (finish - start).total_seconds()
print(f'{programm_length // 3600} hours {programm_length % 3600 // 60} minutes {programm_length % 3600 % 60} seconds')

2026-05-08 15:33:31.201395
2026-05-08 16:56:03.981439
1.0 hours 22.0 minutes 32.78004400000009 seconds


# Работа с архивом полученных текстовых файлов (time_tracks_raw)

In [25]:
# Сжатие архива (если необходимо)
train_zstd_dictionary(
    os.path.join(path, 'time_tracks_raw'),
    os.path.join(path, 'dict_tracks_raw.dict')
)
compress_with_dict(
    os.path.join(path, 'time_tracks_raw'),
    os.path.join(path, 'time_tracks_raw.tar.zst'),
    os.path.join(path, 'dict_tracks_raw.dict')
)

Словарь сохранен в: .\dict_tracks_raw.dict
Архив создан: .\time_tracks_raw.tar.zst


In [26]:
# Распаковка архива
# decompress_with_dict(
#     os.path.join(path, 'time_tracks_raw.tar.zst'),
#     os.path.join(path, 'time_tracks_raw'),
#     os.path.join(path, 'dict_tracks_raw.dict')
# )

# Постобработка

In [27]:
def postprocess_tracks_by_levels(levels, start_dt, end_dt, base_path='.'):    
    """
    Читает 'сырые' файлы трекинга и разбивает их на индивидуальные файлы по ID.
    """
    raw_dir = os.path.join(base_path, 'time_tracks_raw')
    proc_dir = os.path.join(base_path, 'time_tracks_processed')
    
    # if not os.path.exists(proc_dir):
    #     os.makedirs(proc_dir)
    os.makedirs(proc_dir, exist_ok=True)

    for lvl in levels:
        # 1. Формируем имя исходного файла (используя вашу функцию из SI)
        raw_filename = create_result_filename(lvl, start_dt, end_dt)
        raw_path = os.path.join(raw_dir, raw_filename)
        
        if not os.path.exists(raw_path):
            print(f"Файл не найден: {raw_path}")
            continue
            
        print(f"Постобработка уровня {lvl}...")
        
        # 2. Группируем данные по ID трека
        # Читаем файл (формат: Y M D H ID LatMin LonMin LatCent LonCent Pmin Depth Radius)
        df = pd.read_csv(raw_path, sep=' ', header=None, 
                         names=['Y', 'M', 'D', 'H', 'ID', 'LatMin', 'LonMin', 'LatCent', 'LonCent', 'Pmin', 'Depth', 'Radius'])
        
        # 3. Сохраняем каждый трек в отдельный файл
        for track_id, track_data in df.groupby('ID'):
            track_filename = create_processed_track_filename(lvl, track_id)
            track_path = os.path.join(proc_dir, track_filename)
            
            # Сохраняем без индексов и заголовков, чтобы сохранить ваш текстовый формат
            track_data.to_csv(track_path, sep=' ', index=False, header=False)

## С распараллеливанием

In [28]:
start = datetime.now()
print(start)

Parallel(n_jobs=min(len(levels), n_jobs))(    # занимаем максимально возможное количество ядер (в моём случае 7 из 8)
        delayed(postprocess_tracks_by_levels)([lvl], start_time, end_time, base_path='.') for lvl in levels
    )

finish = datetime.now()
print(finish)

programm_length = (finish - start).total_seconds()
print(f'{programm_length // 3600} hours {programm_length % 3600 // 60} minutes {programm_length % 3600 % 60} seconds')

2026-05-08 16:56:14.943796
2026-05-08 17:00:15.145102
0.0 hours 4.0 minutes 0.20130599999998822 seconds


# Работа с архивом постобработанных текстовых файлов (time_tracks_processed)

In [29]:
# Сжатие архива (если необходимо)
train_zstd_dictionary(
    os.path.join(path, 'time_tracks_processed'),
    os.path.join(path, 'dict_tracks_processed.dict')
)
compress_with_dict(
    os.path.join(path, 'time_tracks_processed'),
    os.path.join(path, 'time_tracks_processed.tar.zst'),
    os.path.join(path, 'dict_tracks_processed.dict')
)

Словарь сохранен в: .\dict_tracks_processed.dict
Архив создан: .\time_tracks_processed.tar.zst


In [30]:
# Распаковка архива
# decompress_with_dict(
#     os.path.join(path, 'time_tracks_processed.tar.zst'),
#     os.path.join(path, 'time_tracks_processed'),
#     os.path.join(path, 'dict_tracks_processed.dict')
# )